# Limpieza y transformación

## Librerías y carga del conjunto de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pycountry
import pyarrow

In [2]:
#%pip install --upgrade --force-reinstall pyarrow

In [3]:
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VYPrOu0Vs3I0hKLLjiPGrA/survey-data-with-duplicate.csv')
df.head(2)

,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [4]:
colums_selected = ['ResponseId', 'Age', 'EdLevel', 'RemoteWork', 'YearsCode', 'YearsCodePro', 'Country', 'LanguageHaveWorkedWith', 
                   'LanguageWantToWorkWith', 'LanguageAdmired', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DatabaseAdmired', 
                   'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'PlatformAdmired', 'NEWCollabToolsHaveWorkedWith', 
                   'NEWCollabToolsWantToWorkWith', 'NEWCollabToolsAdmired', 'CompTotal']
df_focus = df[colums_selected]
df_focus.head()

,ResponseId,Age,EdLevel,RemoteWork,YearsCode,YearsCodePro,Country,LanguageHaveWorkedWith,LanguageWantToWorkWith,LanguageAdmired,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,DatabaseAdmired,PlatformHaveWorkedWith,PlatformWantToWorkWith,PlatformAdmired,NEWCollabToolsHaveWorkedWith,NEWCollabToolsWantToWorkWith,NEWCollabToolsAdmired,CompTotal
0,1,Under 18 years old,Primary/elementary school,Remote,NaN,NaN,United States of America,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Remote,20,17,United Kingdom of Great Britain and Northern I...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Dynamodb;MongoDB;PostgreSQL,PostgreSQL,PostgreSQL,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,NaN
2,3,45-54 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Remote,37,27,United Kingdom of Great Britain and Northern I...,C#,C#,C#,Firebase Realtime Database,Firebase Realtime Database,Firebase Realtime Database,Google Cloud,Google Cloud,Google Cloud,Visual Studio,Visual Studio,Visual Studio,NaN
3,4,18-24 years old,Some college/university study without earning ...,NaN,4,NaN,Canada,C;C++;HTML/CSS;Java;JavaScript;PHP;PowerShell;...,HTML/CSS;Java;JavaScript;PowerShell;Python;SQL...,HTML/CSS;Java;JavaScript;PowerShell;Python;SQL...,MongoDB;MySQL;PostgreSQL;SQLite,MongoDB;MySQL;PostgreSQL,MongoDB;MySQL;PostgreSQL,Amazon Web Services (AWS);Fly.io;Heroku,Amazon Web Services (AWS);Vercel,Amazon Web Services (AWS),NaN,NaN,NaN,NaN
4,5,18-24 years old,"Secondary school (e.g. American high school, G...",NaN,9,NaN,Norway,C++;HTML/CSS;JavaScript;Lua;Python;Rust,C++;HTML/CSS;JavaScript;Lua;Python,C++;HTML/CSS;JavaScript;Lua;Python,PostgreSQL;SQLite,PostgreSQL;SQLite,PostgreSQL;SQLite,NaN,NaN,NaN,Vim,Vim,Vim,NaN


In [5]:
(df_focus.isnull().mean() *100).sort_values(ascending=False)

PlatformAdmired                 52.047909
CompTotal                       48.448600
PlatformWantToWorkWith          47.227951
DatabaseAdmired                 41.075821
PlatformHaveWorkedWith          35.258261
DatabaseWantToWorkWith          34.963411
DatabaseHaveWorkedWith          23.203019
NEWCollabToolsAdmired           22.503323
LanguageAdmired                 22.257360
YearsCodePro                    21.134485
NEWCollabToolsWantToWorkWith    20.401179
RemoteWork                      16.250363
LanguageWantToWorkWith          14.800556
NEWCollabToolsHaveWorkedWith    11.989550
Country                          9.942405
LanguageHaveWorkedWith           8.698840
YearsCode                        8.509403
EdLevel                          7.110011
ResponseId                       0.000000
Age                              0.000000
dtype: float64

In [6]:
df_focus = df_focus.drop(columns=['CompTotal'])

## Limpieza

### Eliminación de duplicados

In [7]:
df_focus.duplicated(subset='ResponseId').sum()

np.int64(20)

In [8]:
df_without_duplicated = df_focus.drop_duplicates(subset='ResponseId').copy()
df_without_duplicated.duplicated(subset='ResponseId').sum()

np.int64(0)

### Eliminación de espacios en blanco

In [9]:
df_obj = df_without_duplicated.select_dtypes(include='object').columns

for col in df_obj:
    df_without_duplicated[col] = df_without_duplicated[col].str.strip()

### Eliminación de registros sin información en tecnologías

In [10]:
print('Antes: ', df_without_duplicated['ResponseId'].nunique())

Antes:  65437


In [11]:
cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'LanguageAdmired',
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DatabaseAdmired',
    'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'PlatformAdmired',
    'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'NEWCollabToolsAdmired'
]

df_no_response = df_without_duplicated[df_without_duplicated[cols].isna().all(axis=1)]

df_without_duplicated = df_without_duplicated.drop(df_no_response.index)

print('Después: ', df_without_duplicated['ResponseId'].nunique())

Después:  60059


### Transformación de variables

In [12]:
df_without_duplicated.head(3)

,ResponseId,Age,EdLevel,RemoteWork,YearsCode,YearsCodePro,Country,LanguageHaveWorkedWith,LanguageWantToWorkWith,LanguageAdmired,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,DatabaseAdmired,PlatformHaveWorkedWith,PlatformWantToWorkWith,PlatformAdmired,NEWCollabToolsHaveWorkedWith,NEWCollabToolsWantToWorkWith,NEWCollabToolsAdmired
1,2,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Remote,20,17,United Kingdom of Great Britain and Northern I...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Dynamodb;MongoDB;PostgreSQL,PostgreSQL,PostgreSQL,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm
2,3,45-54 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Remote,37,27,United Kingdom of Great Britain and Northern I...,C#,C#,C#,Firebase Realtime Database,Firebase Realtime Database,Firebase Realtime Database,Google Cloud,Google Cloud,Google Cloud,Visual Studio,Visual Studio,Visual Studio
3,4,18-24 years old,Some college/university study without earning ...,NaN,4,NaN,Canada,C;C++;HTML/CSS;Java;JavaScript;PHP;PowerShell;...,HTML/CSS;Java;JavaScript;PowerShell;Python;SQL...,HTML/CSS;Java;JavaScript;PowerShell;Python;SQL...,MongoDB;MySQL;PostgreSQL;SQLite,MongoDB;MySQL;PostgreSQL,MongoDB;MySQL;PostgreSQL,Amazon Web Services (AWS);Fly.io;Heroku,Amazon Web Services (AWS);Vercel,Amazon Web Services (AWS),NaN,NaN,NaN


In [13]:
df_without_duplicated.dtypes

ResponseId                       int64
Age                             object
EdLevel                         object
RemoteWork                      object
YearsCode                       object
YearsCodePro                    object
Country                         object
LanguageHaveWorkedWith          object
LanguageWantToWorkWith          object
LanguageAdmired                 object
DatabaseHaveWorkedWith          object
DatabaseWantToWorkWith          object
DatabaseAdmired                 object
PlatformHaveWorkedWith          object
PlatformWantToWorkWith          object
PlatformAdmired                 object
NEWCollabToolsHaveWorkedWith    object
NEWCollabToolsWantToWorkWith    object
NEWCollabToolsAdmired           object
dtype: object

### Estandarización y agrupación de atributos categóricos

In [14]:
df_without_duplicated['YearsCode'].unique()

array(['20', '37', '4', '9', '10', '7', '1', '15', '30', '31', '6', '12',
       '22', '5', '36', '25', '44', '24', '18', '3', '8',
       'More than 50 years', '11', '29', '40', '39', nan, '42', '34',
       '19', '2', '35', '16', '33', '13', '23', '14', '28', '17', '21',
       '43', '46', '26', '32', '41', '45', '27', '38', '50', '48', '47',
       'Less than 1 year', '49'], dtype=object)

In [15]:
df_without_duplicated['YearsCodePro'].unique()

array(['17', '27', nan, '7', '11', '25', '12', '10', '3',
       'Less than 1 year', '18', '37', '15', '20', '6', '2', '16', '8',
       '14', '4', '45', '1', '24', '29', '5', '30', '26', '9', '33', '13',
       '35', '23', '31', '19', '21', '28', '34', '32', '22', '40', '50',
       '39', '44', '42', '41', '36', '38', 'More than 50 years', '43',
       '47', '48', '46', '49'], dtype=object)

In [16]:
def cat_to_num(df, cols):
    mapping = {
        'Less than 1 year': 0.5,
        'More than 50 years': 51
    }
    for col in cols:
        df[f'{col}_num'] = (
            df[col].replace(mapping).pipe(pd.to_numeric, errors = 'coerce')
        )

    return df

In [17]:
df_without_duplicated = cat_to_num(df_without_duplicated, ['YearsCode', 'YearsCodePro'])
df_without_duplicated[['YearsCode_num', 'YearsCodePro_num']].dtypes

YearsCode_num       float64
YearsCodePro_num    float64
dtype: object

In [18]:
def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

df_without_duplicated['ISO3'] = df_without_duplicated['Country'].apply(to_iso3)

In [19]:
df_without_duplicated['ISO3'].unique()

array(['GBR', 'CAN', 'NOR', 'USA', 'UZB', 'SRB', 'POL', 'PHL', 'BGR',
       'CHE', 'IND', 'DEU', 'IRL', 'ITA', 'UKR', 'AUS', 'BRA', 'JPN',
       'AUT', None, 'FRA', 'SAU', 'ROU', 'NPL', 'DZA', 'SWE', 'NLD',
       'HRV', 'PAK', 'CZE', 'MKD', 'FIN', 'SVK', 'RUS', 'GRC', 'ISR',
       'BEL', 'MEX', 'TZA', 'HUN', 'ARG', 'PRT', 'LKA', 'LVA', 'CHN',
       'SGP', 'LBN', 'ESP', 'ZAF', 'LTU', 'VNM', 'DOM', 'IDN', 'MAR',
       'TWN', 'GEO', 'SMR', 'TUN', 'BGD', 'NGA', 'LIE', 'DNK', 'ECU',
       'MYS', 'ALB', 'AZE', 'CHL', 'GHA', 'PER', 'BOL', 'EGY', 'LUX',
       'MNE', 'CYP', 'PRY', 'KAZ', 'SVN', 'JOR', 'CRI', 'JAM', 'THA',
       'NIC', 'MMR', 'RWA', 'BIH', 'BEN', 'SLV', 'ZWE', 'AFG', 'EST',
       'MLT', 'URY', 'BLR', 'COL', 'MDA', 'IMN', 'NZL', 'ARM', 'MDV',
       'ETH', 'ARE', 'FJI', 'GTM', 'UGA', 'TKM', 'MUS', 'KEN', 'CUB',
       'GAB', 'BHS', 'KOR', 'ISL', 'HND', 'LAO', 'MNG', 'KHM', 'MDG',
       'AGO', 'SYR', 'IRQ', 'NAM', 'SEN', 'KGZ', 'ZMB', 'CIV', 'KWT',
       'TJK', 'BDI', 

In [20]:
df_without_duplicated[df_without_duplicated['ISO3'].isna()]['Country'].unique()

array(['Iran, Islamic Republic of...', 'Turkey', 'Kosovo',
       'Venezuela, Bolivarian Republic of...', 'Republic of Korea',
       'Nomadic', 'Palestine', 'Hong Kong (S.A.R.)',
       'Democratic Republic of the Congo', 'Swaziland',
       'Congo, Republic of the...', 'Libyan Arab Jamahiriya',
       'Cape Verde', 'Micronesia, Federated States of...', nan],
      dtype=object)

In [21]:
manual_iso_mapping = {
    'Iran, Islamic Republic of...': 'IRN',
    'Turkey': 'TUR',
    'Kosovo': 'XKX',
    'Venezuela, Bolivarian Republic of...': 'VEN',
    'Republic of Korea': 'KOR',
    'Nomadic': None,
    'Palestine': 'PSE',
    'Hong Kong (S.A.R.)': 'HKG',    # región especial
    'Democratic Republic of the Congo': 'COD',
    'Swaziland': 'SWZ',              # Eswatini
    'Congo, Republic of the...': 'COG',
    'Libyan Arab Jamahiriya': 'LBY', # Libya
    'Cape Verde': 'CPV',             # Cabo Verde
    'Micronesia, Federated States of...': 'FSM',
    
}
df_without_duplicated.loc[df_without_duplicated['ISO3'].isna(), 'ISO3'] = (
    df_without_duplicated.loc[df_without_duplicated['ISO3'].isna(), 'Country']
      .replace(manual_iso_mapping)
)
df_without_duplicated[df_without_duplicated['ISO3'].isna()]['Country'].unique()

array(['Nomadic', nan], dtype=object)

In [22]:
df_without_duplicated['EdLevel'].unique()

array(['Bachelor’s degree (B.A., B.S., B.Eng., etc.)',
       'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)',
       'Some college/university study without earning a degree',
       'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)',
       'Primary/elementary school',
       'Professional degree (JD, MD, Ph.D, Ed.D, etc.)',
       'Associate degree (A.A., A.S., etc.)', 'Something else', nan],
      dtype=object)

In [23]:
macro_ed_map = {
    'Primary/elementary school': 'No Formal Degree',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'No Formal Degree',
    'Some college/university study without earning a degree': 'No Formal Degree',
    
    'Associate degree (A.A., A.S., etc.)': 'Undergraduate Degree',
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 'Undergraduate Degree',
    
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 'Postgraduate Degree',
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 'Postgraduate Degree',
    
    'Something else': 'Other'
}

df_without_duplicated['EdLevel_Group'] = df_without_duplicated['EdLevel'].map(macro_ed_map).fillna('Not specified')
df_without_duplicated['EdLevel_Group'].isnull().sum()

np.int64(0)

In [24]:
df_without_duplicated['Age'].unique()

array(['35-44 years old', '45-54 years old', '18-24 years old',
       'Under 18 years old', '25-34 years old', '55-64 years old',
       'Prefer not to say', '65 years or older'], dtype=object)

In [25]:
age_map = {
    'Under 18 years old': 'Under 18',
    '18-24 years old': '18-24',
    '25-34 years old': '25-34',
    '35-44 years old': '35-44',
    '45-54 years old': '45-54',
    '55-64 years old': '55-64',
    '65 years or older': '65+',
    'Prefer not to say': 'Unknown'
}

df_without_duplicated['Age_Group'] = df_without_duplicated['Age'].map(age_map)
df_without_duplicated['Age_Group'].isnull().sum()

np.int64(0)

In [26]:
df_without_duplicated['RemoteWork'].unique()

array(['Remote', nan, 'In-person', 'Hybrid (some remote, some in-person)'],
      dtype=object)

In [27]:
df_without_duplicated['RemoteWork_Clean'] = (df_without_duplicated['RemoteWork']
                                             .replace('Hybrid (some remote, some in-person)', 'Hybrid')
                                             .fillna('Not specified'))
df_without_duplicated['RemoteWork_Clean'].isnull().sum()

np.int64(0)

In [28]:
df_clean = df_without_duplicated.drop(columns=[
    'YearsCode',
    'YearsCodePro',
    'Country',
    'EdLevel',
    'Age',
    'RemoteWork'
]).rename(columns={
    'YearsCode_num': 'YearsCode',
    'YearsCodePro_num': 'YearsCodePro',
    'ISO3': 'Country',
    'EdLevel_grouped': 'EdLevel',
    'RemoteWork_clean': 'RemoteWork'
})

In [29]:
df_clean.columns

Index(['ResponseId', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith',
       'LanguageAdmired', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith',
       'DatabaseAdmired', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith',
       'PlatformAdmired', 'NEWCollabToolsHaveWorkedWith',
       'NEWCollabToolsWantToWorkWith', 'NEWCollabToolsAdmired', 'YearsCode',
       'YearsCodePro', 'Country', 'EdLevel_Group', 'Age_Group',
       'RemoteWork_Clean'],
      dtype='object')

In [30]:
df_clean.isnull().sum().sort_values(ascending = False)

PlatformAdmired                 28682
PlatformWantToWorkWith          25527
DatabaseAdmired                 21502
PlatformHaveWorkedWith          17693
DatabaseWantToWorkWith          17501
YearsCodePro                    10005
DatabaseHaveWorkedWith           9805
NEWCollabToolsAdmired            9348
LanguageAdmired                  9187
NEWCollabToolsWantToWorkWith     7972
LanguageWantToWorkWith           4307
NEWCollabToolsHaveWorkedWith     2467
Country                          2142
YearsCode                        2094
LanguageHaveWorkedWith            314
ResponseId                          0
EdLevel_Group                       0
Age_Group                           0
RemoteWork_Clean                    0
dtype: int64

In [31]:
df_clean.head()

,ResponseId,LanguageHaveWorkedWith,LanguageWantToWorkWith,LanguageAdmired,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,DatabaseAdmired,PlatformHaveWorkedWith,PlatformWantToWorkWith,PlatformAdmired,NEWCollabToolsHaveWorkedWith,NEWCollabToolsWantToWorkWith,NEWCollabToolsAdmired,YearsCode,YearsCodePro,Country,EdLevel_Group,Age_Group,RemoteWork_Clean
1,2,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Dynamodb;MongoDB;PostgreSQL,PostgreSQL,PostgreSQL,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,20.0,17.0,GBR,Undergraduate Degree,35-44,Remote
2,3,C#,C#,C#,Firebase Realtime Database,Firebase Realtime Database,Firebase Realtime Database,Google Cloud,Google Cloud,Google Cloud,Visual Studio,Visual Studio,Visual Studio,37.0,27.0,GBR,Postgraduate Degree,45-54,Remote
3,4,C;C++;HTML/CSS;Java;JavaScript;PHP;PowerShell;...,HTML/CSS;Java;JavaScript;PowerShell;Python;SQL...,HTML/CSS;Java;JavaScript;PowerShell;Python;SQL...,MongoDB;MySQL;PostgreSQL;SQLite,MongoDB;MySQL;PostgreSQL,MongoDB;MySQL;PostgreSQL,Amazon Web Services (AWS);Fly.io;Heroku,Amazon Web Services (AWS);Vercel,Amazon Web Services (AWS),NaN,NaN,NaN,4.0,NaN,CAN,No Formal Degree,18-24,Not specified
4,5,C++;HTML/CSS;JavaScript;Lua;Python;Rust,C++;HTML/CSS;JavaScript;Lua;Python,C++;HTML/CSS;JavaScript;Lua;Python,PostgreSQL;SQLite,PostgreSQL;SQLite,PostgreSQL;SQLite,NaN,NaN,NaN,Vim,Vim,Vim,9.0,NaN,NOR,No Formal Degree,18-24,Not specified
5,6,Bash/Shell (all shells);HTML/CSS;Java;JavaScri...,Bash/Shell (all shells);HTML/CSS;Java;JavaScri...,Bash/Shell (all shells);HTML/CSS;Java;JavaScri...,Cloud Firestore,Cloud Firestore,Cloud Firestore,Cloudflare,Cloudflare,Cloudflare,Nano;Vim;Visual Studio Code;Xcode,Nano;Vim;Visual Studio Code;Xcode,Nano;Vim;Visual Studio Code;Xcode,10.0,NaN,USA,No Formal Degree,Under 18,Not specified


### Variables multirespuesta

In [32]:
def explode_multiresponse(df, column, relation_name, entity_name, delimiter=';'):
    temp = df[['ResponseId', column]].dropna().copy()
    temp[column] = temp[column].str.split(delimiter)
    temp = temp.explode(column)
    temp[column] = temp[column].str.strip()
    
    temp = temp.rename(columns={column: entity_name})
    temp['Relation'] = relation_name
    
    return temp

In [33]:
def compute_relation_counts(df, entity_name, total_resp):
    temp = pd.crosstab(df[entity_name], df['Relation'])
    temp['InterestRatio'] = (temp['WantToWorkWith'] / temp['HaveWorkedWith'])
    temp['AdmiredRatio'] = (temp['Admired'] / temp['HaveWorkedWith'])
    temp['UsageRate'] = (temp['HaveWorkedWith'] / total_resp)
    temp['InterestRate'] = (temp['WantToWorkWith'] / total_resp)
    temp['AdmiredRate'] = (temp['Admired'] / total_resp)
    temp['GrowthPotential'] = (temp['WantToWorkWith'] - temp['HaveWorkedWith'])
    temp['NetInterest'] = ((temp['WantToWorkWith'] - temp['HaveWorkedWith']) / temp['HaveWorkedWith'])
    temp['ConversionPotential'] = (temp['WantToWorkWith'] / (total_resp - temp['HaveWorkedWith']))

    temp = temp.replace([np.inf, -np.inf], np.nan)
    
    return temp

In [34]:
languages = pd.concat([
    explode_multiresponse(df_without_duplicated, 'LanguageHaveWorkedWith', 'HaveWorkedWith', 'Language'),
    explode_multiresponse(df_without_duplicated, 'LanguageWantToWorkWith', 'WantToWorkWith', 'Language'),
    explode_multiresponse(df_without_duplicated, 'LanguageAdmired', 'Admired', 'Language')
], ignore_index=True)

databases = pd.concat([
    explode_multiresponse(df_without_duplicated, 'DatabaseHaveWorkedWith', 'HaveWorkedWith', 'Database'),
    explode_multiresponse(df_without_duplicated, 'DatabaseWantToWorkWith', 'WantToWorkWith', 'Database'),
    explode_multiresponse(df_without_duplicated, 'DatabaseAdmired', 'Admired', 'Database')
], ignore_index = True)

platforms = pd.concat([
    explode_multiresponse(df, 'PlatformHaveWorkedWith', 'HaveWorkedWith', 'Platform'),
    explode_multiresponse(df, 'PlatformWantToWorkWith', 'WantToWorkWith', 'Platform'),
    explode_multiresponse(df, 'PlatformAdmired', 'Admired', 'Platform')
], ignore_index = True)

collab_tools = pd.concat([
    explode_multiresponse(df, 'NEWCollabToolsHaveWorkedWith', 'HaveWorkedWith', 'Collab Tools'),
    explode_multiresponse(df, 'NEWCollabToolsWantToWorkWith', 'WantToWorkWith', 'Collab Tools'),
    explode_multiresponse(df, 'NEWCollabToolsAdmired', 'Admired', 'Collab Tools')
], ignore_index = True)

In [35]:
languages.to_parquet('datos/languages_long.parquet', index = False)
databases.to_parquet('datos/databases.parquet', index = False)
platforms.to_parquet('datos/platforms.parquet', index = False)
collab_tools.to_parquet('datos/collabtools.parquet', index = False)

In [36]:
total_lang = languages['ResponseId'].nunique()
total_db = databases['ResponseId'].nunique()
total_pf = platforms['ResponseId'].nunique()
total_ct = collab_tools['ResponseId'].nunique()

In [37]:
languages_metrics = compute_relation_counts(languages, 'Language', total_lang)
languages_metrics.to_parquet('datos/languages_metrics.parquet')

databases_metrics = compute_relation_counts(databases, 'Database', total_db)
databases_metrics.to_parquet('datos/databases_metrics.parquet')

platforms_metrics = compute_relation_counts(platforms, 'Platform', total_pf)
platforms_metrics.to_parquet('datos/platforms_metrics.parquet')

collab_tools_metrics = compute_relation_counts(collab_tools, 'Collab Tools', total_ct)
collab_tools_metrics.to_parquet('datos/collab_tools_metrics.parquet')

In [38]:
df_clean.to_parquet('datos/respondents.parquet', index=False)
df_clean.to_csv('datos/respondents.csv', index=False)